In [150]:
# Load Packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
import matplotlib.pyplot as plt
import math

In [158]:
#Common functions for all programs 

#The following function finds the date corresponding to the last thursday of a month in a year
#Input: Year, month - both integers
#Output: Python date of last thursday of the month
def find_last_thurs_date_of_month(year, month):
    calendar_of_the_month = calendar.monthcalendar(year, month)
    no_of_weeks = len(calendar_of_the_month)
    if (calendar_of_the_month[no_of_weeks-1][3] != 0):
        return date(year, month, calendar_of_the_month[no_of_weeks-1][3])
    else:
        return date(year, month, calendar_of_the_month[no_of_weeks-2][3])

#Function for getting previous weekday of a date
#Input: any python date
#Output: Previous weekday python date
def prev_weekday(date):
    if (date.weekday() == 0):
        return date - timedelta(days=3)
    elif (date.weekday() == 6):
        return date - timedelta(days=2)
    else:
        return date - timedelta(days=1)
    
#0-Mon, 1-Tues, 2-Wed, 3-Thurs, 4-Fri, 5-Sat, 6-Sun
def next_weekday(date):
    if (date.weekday() == 4):
        return date + timedelta(days=3)
    elif (date.weekday() == 5):
        return date + timedelta(days=2)
    else:
        return date + timedelta(days=1)   

#Function to Check for holidays- if holiday, the closing/expiry day is the previous working_day
#if holiday, select next working day
#Input: python date, holiday python list
#Output: Previous working python 
def prev_workday_if_holiday(date, holidays_list):
    if (date in holidays_list):
        return prev_workday_if_holiday(prev_weekday(date), holidays_list)
    else:
        return date

def next_workday(date, holidays_list):
    if (next_weekday(date) in holidays_list):
        return next_workday(next_weekday(date), holidays_list)
    else:
        return next_weekday(date)    
    
#The following function finds the date corresponding to the previous thursdays
#Input: Date - Python date format, No.of. thursdays back
#Output: The interested thursday python date
def date_of_prev_thurs(date, no_of_thurs_back):
    if (date.weekday() > 3):
        minus_days = date.weekday() - 3
        prev_thurs = date - timedelta(days=minus_days) -timedelta(days = 7 * (no_of_thurs_back-1))
        return prev_thurs
    else:
        prev_thurs = date - timedelta(days=no_of_thurs_back * 7)
        add_days = 3-prev_thurs.weekday()
        return (prev_thurs + timedelta(days=add_days))
    

# The function helps in finding the list option last thursday expiries of the month 
#(we are interest to hedge options of this expiry)
# Input: Dictionary with keys as years and each key has value of list of months we are interested
# Output: List of monthly last thursday expiries
def get_all_monthly_option_expiries(periods_of_interest, holidays_list):
    monthly_expiries = []
    for key in periods_of_interest.keys():
        list_of_months = periods_of_interest[key]
        expiry_dates = [find_last_thurs_date_of_month(key, month) for month in list_of_months]
        expiry_dates = [prev_workday_if_holiday(date, holidays_list) for date in expiry_dates]
        monthly_expiries = monthly_expiries + expiry_dates
    return monthly_expiries

#The following function finds the date corresponding to the previous thursdays
#Input: Date - Python date format, No.of. thursdays back
#Output: The interested thursday python date
def date_of_prev_thurs(date, no_of_thurs_back):
    if (date.weekday() > 3):
        minus_days = date.weekday() - 3
        prev_thurs = date - timedelta(days=minus_days) -timedelta(days = 7 * (no_of_thurs_back-1))
        return prev_thurs
    else:
        prev_thurs = date - timedelta(days=no_of_thurs_back * 7)
        add_days = 3-prev_thurs.weekday()
        return (prev_thurs + timedelta(days=add_days))

#Function to find all weekly expiry option dates for every month of interest (i.e. every thursdays)
#Input: List python monthly expiries
#Output: dictionary - {'monthly_expiry_date:': [4weeks before thurs, 3w, 2w, 1w]}
def find_wkly_expries(mthly_expiries, holidays_list):
    weekly_expiries_dict = {}
    for expiry_date in mthly_expiries:
        wkly_expiries_list = [date_of_prev_thurs(expiry_date, no_of_thurs) for no_of_thurs in range(4, 0, -1)]
        wkly_expiries_list = [prev_workday_if_holiday(date, holidays_list) for date in wkly_expiries_list]
        weekly_expiries_dict[expiry_date.strftime("%d-%b-%Y")] = wkly_expiries_list
    return weekly_expiries_dict

def generate_daily_dates_each_month(dict_wkly_expiries_each_month, holidays_list):
    dict_daily_dates = {}
    for each_month in dict_wkly_expiries_each_month:
        start_date = dict_wkly_expiries_each_month[each_month][0]
        end_date = datetime.strptime(each_month, "%d-%b-%Y").date()
        curr_date = start_date
        monthly_list = []
        while (curr_date < end_date):
            curr_date = next_workday(curr_date, holidays_list)
            monthly_list.append(curr_date)
        dict_daily_dates[each_month] = monthly_list
    return dict_daily_dates

#Load dataframe from csv file of the dats for that month:
#Input:
#     Expiry dates list of options to be hedged in python date format
#     prod_types of the expiries = "CE", "PE", "FUT"
#Output:
#      dataframe with keys as same name as csv file
def load_all_mthly_data(monthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = "mkt_data_covid_region/"):
    df_dict = dict()
    for expiry_prod in list(itertools.product(monthly_expiries_list, prod_type_lists)):
        start_date = date_of_prev_thurs(expiry_prod[0], no_of_thurs_back=4)
        if (expiry_prod[1] == "FUT"):
            file_name = expiry_prod[1] + "IDX_BANKNIFTY_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
        else:
            file_name = "OPTIDX_BANKNIFTY_" + expiry_prod[1] + "_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
        df_dict["df" + "_" + file_name[:-4]] = pd.read_csv(relative_path + file_name)
    return (df_dict)

def call_delta_with_futures(s_t, k, r, vol, dt):
    d_1_c = (1 / (vol * (dt ** 0.5))) 
    d1_log =  (np.log((s_t)/(k)) + ((r + (vol ** 2) / 2) * dt))
    d_1 = d_1_c * d1_log
    dc_ds = norm.cdf(d_1)
    df_ds = np.exp(r * dt)
    dc_df = float(dc_ds/df_ds)
    return dc_df

def find_week(date, holidays_list):
    prev_thurs = date_of_prev_thurs(date, 1)
    week = prev_workday_if_holiday(prev_thurs, holidays_list)
    return week

def create_df_wk_days(dict_daily_dates_error_cal, dict_wkly_expiries_each_month, holidays_list):
    day_wk ={}
    full_days_list = []
    full_week_list = []
    for each_month in dict_daily_dates_error_cal.keys():
        days_list = dict_daily_dates_error_cal[each_month]
        week_list = [find_week(each_day, holidays_list) for each_day in days_list]
        
        days_list = [day.strftime("%d-%b-%Y") for day in days_list]
        week_list = [week.strftime("%d-%b-%Y") for week in week_list]
        
        full_days_list = full_days_list + days_list
        full_week_list = full_week_list + week_list
    day_wk = {"Week": full_week_list, "Day": full_days_list}
    df_wk_day_map = pd.DataFrame(day_wk)
    return df_wk_day_map

# Funtion to get monthly data_frame names stored for each mkt_data from expiry_date
#Input: date, product type is "CE", "PE" or "FUT"
#Output: dataframe string name
def get_mthly_df_name_from_expiry(expiry_date, prod_type, holidays_list):
    start_date = date_of_prev_thurs(expiry_date, no_of_thurs_back=4)
    start_date = prev_workday_if_holiday(start_date, holidays_list)
    if (prod_type == "FUT"):
        df_name_str = "df_" + prod_type + "IDX_BANKNIFTY_" \
                    + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                    + expiry_date.strftime("%d-%b-%Y")
    else:
            df_name_str = "df_" + "OPTIDX_BANKNIFTY_" + prod_type + "_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_date.strftime("%d-%b-%Y")      
    return df_name_str

def generate_dict_opt_value(dict_daily_dates_error_cal,monthly_mkt_data, holidays_list, \
                            prod_type="CE", relative_path="mkt_data_covid_region/"):
    dict_monthly_option_value = {}
    full_dates_list = []
    full_price_list = []
    df_strikes_hedged = pd.read_csv(relative_path + "code_outputs/B0001_" + prod_type + "_strikes_hedged.csv")
    for each_month in dict_daily_dates_error_cal.keys():
        each_month_data = monthly_mkt_data[get_mthly_df_name_from_expiry(datetime.strptime(each_month, "%d-%b-%Y").date(), prod_type, holidays_list)]
        strike = df_strikes_hedged[df_strikes_hedged["Date"] == each_month]["Strike"].iloc[0]
        df_opt_values = each_month_data[each_month_data["Strike Price"] == strike]
        df_opt_values.assign(x=pd.to_datetime(df_opt_values['Date'])).sort_values('x').drop('x', 1)
        dates_list = df_opt_values["Date"].tolist()[1:]
        full_dates_list = full_dates_list + dates_list
        price_list = df_opt_values["Close"].tolist()[1:]
        full_price_list = full_price_list + price_list
    
    dict_monthly_option_value = {"Date": full_dates_list , "Close Price": full_price_list}
    df_monthly_option_value = pd.DataFrame(dict_monthly_option_value)
    
    return df_monthly_option_value

def combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type="CE",\
                             relative_path="mkt_data_covid_region/"):
    list_of_weekly_df = []
    for each_month in dict_wkly_expiries_each_month.keys():
        for week in range(0,4):
            if (week < 3):
                start_date = dict_wkly_expiries_each_month[each_month][week].strftime("%d-%b-%Y") 
                expiry_date = dict_wkly_expiries_each_month[each_month][week+1].strftime("%d-%b-%Y")
            else:
                start_date = dict_wkly_expiries_each_month[each_month][week].strftime("%d-%b-%Y") 
                expiry_date = each_month
            
            file_name = "OPTIDX_BANKNIFTY_" + prod_type + "_" \
                        + start_date +  "_TO_" \
                        + expiry_date + ".csv" 
            df = pd.read_csv(relative_path + file_name)
            list_of_weekly_df.append(df)
    
    weekly_df_prod = pd.concat(list_of_weekly_df, axis=0) 
    return weekly_df_prod
            
  

In [156]:
def find_daily_static_hedging_error(df_wk_day_map, df_ce_weekly, df_pe_weekly,\
                                    df_monthly_option_value,\
                                    prod_type="CE", prod_moneyness="ATM", \
                                    relative_path="mkt_data_covid_region/"):

    df_ce_map = pd.merge(df_wk_day_map, df_ce_weekly, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_pe_map = pd.merge(df_wk_day_map, df_pe_weekly, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_ce_map = df_ce_map[['Week', 'Day', 'Strike Price', 'Close']]
    df_pe_map = df_pe_map[['Week', 'Day', 'Strike Price', 'Close']]
    
    ce_weights = pd.read_csv(relative_path + "code_outputs/B0001_" + prod_moneyness + "_CE_Pfl_weights.csv")
    pe_weights = pd.read_csv(relative_path  + "code_outputs/B0001_" + prod_moneyness + "_PE_Pfl_weights.csv")
    cash_invst = pd.read_csv(relative_path  + "code_outputs/B0001_" + prod_moneyness + "_Cash_Pfl_weights.csv")
    
    df_ce_pfl = pd.merge(df_ce_map, ce_weights, how = 'left', \
                         left_on = ['Week', 'Strike Price'], \
                         right_on = ['Date', 'Strike'])
   
    df_ce_pfl = df_ce_pfl[['Week', 'Day', 'Strike Price', 'Close', 'Pfl_weights']]
    df_ce_pfl['hedge_value'] = df_ce_pfl['Close'] * df_ce_pfl['Pfl_weights']  
    df_ce_pfl = df_ce_pfl.groupby('Day')[['hedge_value']].sum().reset_index()
    df_ce_pfl = df_ce_pfl.assign(x=pd.to_datetime(df_ce_pfl['Day'])).sort_values('x').drop('x', 1)
    
    
    
    df_pe_pfl = pd.merge(df_pe_map, pe_weights, how = 'left', \
                         left_on = ['Week', 'Strike Price'], \
                         right_on = ['Date', 'Strike'])
    df_pe_pfl = df_pe_pfl[['Week', 'Day', 'Strike Price', 'Close', 'Pfl_weights']]
    df_pe_pfl['hedge_value'] = df_pe_pfl['Close'] * df_pe_pfl['Pfl_weights']  
    df_pe_pfl = df_pe_pfl.groupby('Day')[['hedge_value']].sum().reset_index()
    df_pe_pfl = df_pe_pfl.assign(x=pd.to_datetime(df_pe_pfl['Day'])).sort_values('x').drop('x', 1)
    
    ######################################################################
    #Constant Interest rate to be addressed later after discussion wit sir
    ######################################################################
    r = 0.03
    df_cash_pfl = pd.merge(df_wk_day_map, cash_invst, how = 'left', \
                         left_on = ['Week'], \
                         right_on = ['Date'])
    df_cash_pfl = df_cash_pfl[['Week', 'Day', 'Pfl_weights']]
    df_cash_pfl['time_value'] = (pd.to_datetime(df_cash_pfl['Day'], format= "%d-%b-%Y") - pd.to_datetime(df_cash_pfl['Week'], format= "%d-%b-%Y")).dt.days / 365
    df_cash_pfl['Factor'] = df_cash_pfl['time_value'].apply(lambda x: np.exp(x * r))
    df_cash_pfl['hedge_value'] = df_cash_pfl['Factor'] * df_cash_pfl['Pfl_weights']
    
    
    df_static_hedge = df_ce_pfl.merge(df_pe_pfl , how = 'left', left_on='Day', right_on='Day',\
                                suffixes=('_CE', '_PE'))
    df_static_hedge = df_static_hedge.merge(df_cash_pfl,  how = 'left', left_on='Day', right_on='Day')
    df_static_hedge['Hedge_Pfl_value'] = df_static_hedge['hedge_value_CE'] + \
                                         df_static_hedge['hedge_value_PE'] + \
                                         df_static_hedge['hedge_value']
    
    df_static_hedge = df_static_hedge[['Day', 'Hedge_Pfl_value']]
    df_static_hedge = pd.merge(df_static_hedge, df_monthly_option_value, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_static_hedge['Static Hedge Error (BS)'] = df_static_hedge['Hedge_Pfl_value'] - df_static_hedge['Close Price']
    
    df_static_hedge = df_static_hedge[['Date', 'Close Price', 'Hedge_Pfl_value', 'Static Hedge Error (BS)']]
    df_static_hedge.to_csv(relative_path + "code_outputs/C0001_" + prod_type + "_" + prod_moneyness + "_"+ "BS_static_hedge_error.csv", index = False)
    
    return df_static_hedge



In [159]:
#Compute Daily Static  Hedging Errors

###################################
# Common Variables initialisation #
###################################

#Path to refer data
source_path = "mkt_data_covid_region/"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]

#Find the monthly strike of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)
dict_daily_dates_error_cal = generate_daily_dates_each_month(dict_wkly_expiries_each_month, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = source_path)

#Load monthly options' strikes hedged
df_hedged_prod = pd.read_csv(source_path + "code_outputs/B0001_" + prod_type + "_strikes_hedged.csv")

#Load Pfl_weights
df_ce_pfl_weights = pd.read_csv(source_path + "code_outputs/B0001_" + prod_moneyness +  "_CE_Pfl_weights.csv")
df_pe_pfl_weights = pd.read_csv(source_path + "code_outputs/B0001_" + prod_moneyness +  "_PE_Pfl_weights.csv")
df_cash_pfl_weights = pd.read_csv(source_path + "code_outputs/B0001_" + prod_moneyness +  "_Cash_Pfl_weights.csv")

df_wk_day_map = create_df_wk_days(dict_daily_dates_error_cal, dict_wkly_expiries_each_month, holidays_list)


df_monthly_option_value = generate_dict_opt_value(dict_daily_dates_error_cal,mthly_mkt_data, holidays_list, \
                          prod_type=prod_type, relative_path=source_path)


df_ce_weekly = combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type="CE", relative_path=source_path)
df_pe_weekly = combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type="PE", relative_path=source_path)

df_static_hedge = find_daily_static_hedging_error(df_wk_day_map, df_ce_weekly, df_pe_weekly, df_monthly_option_value, \
                                           prod_type=prod_type, prod_moneyness=prod_moneyness)


#Final output
#DataFrame: "Date", "Option Value", "Quantity" - this to be considered later, "Static Hedging Error", "Dynamic Hedging Error"
